# 🐳 BI Docker Workshop — Repository Walkthrough (File-by-File)

This notebook explains **every file in the repository** and how they work together to deliver a mini Business Intelligence stack:

- **PostgreSQL** database seeded with a BI-friendly dataset
- **FastAPI** web service exposing endpoints (easy to ingest from Power BI)
- **Docker Compose** to run everything with one command
- **VS Code Dev Container** configuration so students get a consistent environment

> Target audience: Master 2 Data / BI students  
> Goal: understand *what each file does*, *why it exists*, and *how to modify it safely*.


## 0) Repository tree
Let’s start by printing the project tree so you can locate each file.

In [ ]:

from pathlib import Path

ROOT = Path(".")  # Run this cell from the repo root in VS Code / Dev Container

def print_tree(root: Path, max_depth: int = 4):
    root = root.resolve()
    for path in sorted(root.rglob("*")):
        rel = path.relative_to(root)
        depth = len(rel.parts)
        if depth > max_depth:
            continue
        indent = "  " * (depth - 1)
        if path.is_dir():
            print(f"{indent}📁 {rel}/")
        else:
            print(f"{indent}📄 {rel}")

print_tree(ROOT, max_depth=6)


📁 .devcontainer/
  📄 .devcontainer\devcontainer.json
📄 .env
📁 .git/
  📄 .git\config
  📄 .git\description
  📄 .git\HEAD
  📁 .git\hooks/
    📄 .git\hooks\applypatch-msg.sample
    📄 .git\hooks\commit-msg.sample
    📄 .git\hooks\fsmonitor-watchman.sample
    📄 .git\hooks\post-update.sample
    📄 .git\hooks\pre-applypatch.sample
    📄 .git\hooks\pre-commit.sample
    📄 .git\hooks\pre-merge-commit.sample
    📄 .git\hooks\pre-push.sample
    📄 .git\hooks\pre-rebase.sample
    📄 .git\hooks\pre-receive.sample
    📄 .git\hooks\prepare-commit-msg.sample
    📄 .git\hooks\push-to-checkout.sample
    📄 .git\hooks\update.sample
  📁 .git\info/
    📄 .git\info\exclude
  📁 .git\objects/
    📁 .git\objects\info/
    📁 .git\objects\pack/
  📁 .git\refs/
    📁 .git\refs\heads/
    📁 .git\refs\tags/
📄 .gitignore
📁 api/
  📄 api\Dockerfile
  📄 api\main.py
  📄 api\requirements.txt
📄 bi-docker-workshop_explained.ipynb
📁 database/
  📄 database\init.sql
📄 docker-compose.yml
📁 docs/
  📄 docs\bigquery_extension.md


## 1) How the pieces fit together

### The runtime architecture
1. **`docker-compose.yml`** starts **two services**:
   - `postgres` (database)  
   - `api` (FastAPI app that queries Postgres and returns JSON)

2. On first boot, Postgres runs **`database/init.sql`** automatically to create and seed tables.

3. The API reads its DB connection parameters from environment variables (from Compose).

4. (Optional but recommended) **`.devcontainer/devcontainer.json`** tells VS Code how to open this repo inside the running `api` container so every student has the same Python/OS environment.

### Why this matters for BI
Power BI / Tableau / Excel can ingest:
- **Databases** (Postgres connector)  
- **Web APIs** (JSON endpoints)  

This workshop intentionally provides *both* to teach two common integration paths.


## 2) Quickstart commands (what students run)

These are the typical commands you use during the workshop.


In [ ]:

# 1) Start the stack
# (Run in terminal, not Python)
# docker compose up --build

# 2) Test API locally (once it's running)
# curl http://localhost:18000/health
# curl "http://localhost:18000/sales?limit=10"

# 3) Stop everything
# docker compose down


# 3) File-by-file explanation

## 📄 `.devcontainer/devcontainer.json`

### What it does  
Configures **VS Code Dev Containers** so VS Code can “attach” to the running container as a full dev environment.

### Why it’s important (teaching)  
- Everyone codes in the same Linux environment  
- Same Python version, same dependencies  
- Avoids Windows vs macOS differences (huge in classrooms)

### Key fields  
- `dockerComposeFile`: tells VS Code to use your Compose stack  
- `service`: which service becomes the dev environment (`api`)  
- `workspaceFolder`: where the repo is mounted inside the container  
- `extensions`: recommended VS Code extensions for students


### File content (for reference)

<details><summary>Click to expand</summary>

```json
{
  "name": "bi-docker-workshop",
  "dockerComposeFile": ["../docker-compose.yml"],
  "service": "api",
  "workspaceFolder": "/workspaces/bi-docker-workshop",
  "customizations": {
    "vscode": {
      "extensions": [
        "ms-azuretools.vscode-docker",
        "ms-python.python"
      ]
    }
  }
}
```

</details>

## 📄 `.env`

### What it does  
Defines environment variables (credentials/config) in a standard key=value format.

### Why it’s important  
In production, you rarely hardcode credentials in Compose.  
This repo keeps it simple for teaching, but `.env` is the right pattern.

> ⚠️ Never commit real secrets. For workshops, dummy credentials are fine.


### File content (for reference)

<details><summary>Click to expand</summary>

```bash
# Kept for teaching purposes (Compose currently uses inline env vars)
POSTGRES_DB=sales
POSTGRES_USER=admin
POSTGRES_PASSWORD=admin
```

</details>

## 📄 `.gitignore`

### What it does  
Tells Git which files should **not** be committed (caches, secrets, generated files).

### Why it’s important  
- Prevents leaking credentials  
- Keeps the repo clean  
- Avoids committing huge/irrelevant artifacts


### File content (for reference)

<details><summary>Click to expand</summary>

```
.DS_Store
__pycache__/
*.pyc
.env.local
.vscode/
```

</details>

## 📄 `README.md`

### What it does  
Entry point documentation: how to run the project, what it contains, and the workshop goals.

### Why it’s important  
Students will read this first. In real projects, `README.md` is the “front door”.


### File content (for reference)

<details><summary>Click to expand</summary>

```markdown
# BI Docker Workshop — Docker → Postgres/API → Power BI (+ BigQuery extension)

This repo is designed for a **3–4 hour Master 2 BI/Data** workshop.

## What students build
A reproducible BI backend (PostgreSQL + FastAPI) with Docker Compose, then connect:
- **Power BI Desktop (Windows)** → **PostgreSQL** (recommended)
- Power BI Desktop → **REST API** (Web connector)
- (Optional) Export to **BigQuery** for the “modern stack” extension

---

## 0) Prerequisites

### Windows users
- Docker Desktop
- VS Code + Docker extension

### macOS users (Power BI Desktop constraint)
Power BI Desktop is Windows-only.
You have two workable setups:

**Option A (recommended):**
- Docker Desktop on macOS
- Parallels (Windows 11) for Power BI Desktop

**Option B:**
- Use Power BI Service in browser (less ideal for teaching modeling)

---

## 1) Quickstart (everyone)

### 1. Clone
```bash
git clone https://github.com/<YOUR_ORG>/bi-docker-workshop.git
cd bi-docker-workshop
```

### 2. Run the stack
```bash
docker compose up --build
```

You should have:
- PostgreSQL on `localhost:5432`
- API on `http://localhost:8000`

### 3. Test the API
Open:
- `http://localhost:8000/health`
- `http://localhost:8000/sales`

---

## 2) Power BI Desktop connections

### A) PostgreSQL connector (best for BI)
In Power BI Desktop → **Get data** → **PostgreSQL**:

**Windows (native):**
- Server: `localhost`
- Port: `5432`
- Database: `sales`
- Username: `admin`
- Password: `admin`

**macOS (Power BI running inside Windows VM):**
Use the VM-safe hostname:
- Server: `host.docker.internal`
- Port: `5432`

> If you run Docker inside Windows (instead of macOS), then use `localhost`.

### B) REST API (Web)
Power BI Desktop → **Get data** → **Web**:
- `http://localhost:8000/sales`

or in a Windows VM on macOS:
- `http://host.docker.internal:8000/sales`

---

## 3) VS Code Dev Containers (optional)
This repo includes a `.devcontainer` config so students can open the project in a containerized dev environment in VS Code.

See `docs/devcontainer.md`.

---

## 4) BigQuery extension (optional)
See `docs/bigquery_extension.md` for a clean “Docker compute → BigQuery storage → Power BI semantic layer” extension.

---
...
(Truncated in notebook — open the file in VS Code to read the full content.)
```

</details>

## 📄 `api/Dockerfile`

### What it does  
Builds the **API container image** (Python runtime + dependencies + your code).

### Why it’s important  
- Guarantees the API runs with the right Python and libraries  
- Separates “build time” (install deps) from “runtime” (run the server)  
- The Dockerfile is the *recipe*; the image is the *result*

### Things students usually edit  
- Add OS packages (e.g., `curl`)  
- Change Python version base image  
- Optimize with caching (copy `requirements.txt` first, then install)


### File content (for reference)

<details><summary>Click to expand</summary>

```dockerfile
FROM python:3.11-slim

RUN apt-get update && apt-get install -y --no-install-recommends build-essential \
 && rm -rf /var/lib/apt/lists/*

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY main.py .

EXPOSE 8000
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
```

</details>

## 📄 `api/main.py`

### What it does  
Implements a **FastAPI** application exposing endpoints that query Postgres.

### Why it’s important  
- BI tools love simple JSON endpoints (Power BI Web connector)  
- Demonstrates “data access layer” + “API layer” separation  
- Teaches environment variables for configuration

### Key concepts inside  
- `get_conn()` reads DB settings from env vars  
- `/health` endpoint for monitoring & debugging  
- `/sales` endpoint returns table rows with a `limit` parameter  
- `/kpi/revenue_by_country` shows a BI-style aggregation endpoint


### File content (for reference)

<details><summary>Click to expand</summary>

```python
import os
from typing import List, Dict, Any
from fastapi import FastAPI
import psycopg2
from psycopg2.extras import RealDictCursor

app = FastAPI(title="BI Docker Workshop API", version="1.0.0")

def get_conn():
    return psycopg2.connect(
        host=os.getenv("DB_HOST", "postgres"),
        port=int(os.getenv("DB_PORT", "5432")),
        dbname=os.getenv("DB_NAME", "sales"),
        user=os.getenv("DB_USER", "admin"),
        password=os.getenv("DB_PASSWORD", "admin"),
        cursor_factory=RealDictCursor,
    )

@app.get("/health")
def health() -> Dict[str, str]:
    try:
        with get_conn() as conn:
            with conn.cursor() as cur:
                cur.execute("SELECT 1 as ok;")
                _ = cur.fetchone()
        return {"status": "ok"}
    except Exception as e:
        return {"status": "error", "detail": str(e)}

@app.get("/sales")
def sales(limit: int = 200) -> List[Dict[str, Any]]:
    """Return sales rows (Power BI can ingest this via Web connector)."""
    query = """
        SELECT sale_date, country, channel, product_category, product,
               units, unit_price, revenue
        FROM sales
        ORDER BY sale_date DESC
        LIMIT %s;
    """
    with get_conn() as conn:
        with conn.cursor() as cur:
            cur.execute(query, (limit,))
            rows = cur.fetchall()
    return rows

@app.get("/kpi/revenue_by_country")
def revenue_by_country() -> List[Dict[str, Any]]:
    query = """
        SELECT country, SUM(revenue) AS total_revenue
        FROM sales
        GROUP BY country
        ORDER BY total_revenue DESC;
    """
    with get_conn() as conn:
        with conn.cursor() as cur:
            cur.execute(query)
            rows = cur.fetchall()
    return rows
```

</details>

## 📄 `api/requirements.txt`

### What it does  
Pins the Python dependencies for the API service.

### Why it’s important  
- Reproducible installs (same versions for everyone)  
- Avoids “dependency drift” between students’ machines

### Common edits  
- Add `pandas` for data shaping  
- Add `sqlalchemy` for ORM usage  
- Pin versions more strictly for production


### File content (for reference)

<details><summary>Click to expand</summary>

```
fastapi==0.115.6
uvicorn[standard]==0.32.1
psycopg2-binary==2.9.9
pandas==2.2.3
```

</details>

## 📄 `database/init.sql`

### What it does  
Creates tables and **seeds the database** with initial data.

### Why it’s important  
- Students get data immediately (no extra data download)  
- Enables BI dashboards on day 1  
- Shows the standard “init scripts” approach in Dockerized Postgres

### How it runs  
Because `docker-compose.yml` mounts this file into:
`/docker-entrypoint-initdb.d/`  
Postgres runs it automatically **only when the database is created the first time**.

> If you change this script and want it to re-run, you usually need to remove the volume (`pgdata`) and recreate.


### File content (for reference)

<details><summary>Click to expand</summary>

```sql
-- Simple but BI-realistic dataset: daily sales by country/product with seasonality + promos
CREATE TABLE IF NOT EXISTS sales (
    sale_date DATE NOT NULL,
    country TEXT NOT NULL,
    channel TEXT NOT NULL,
    product_category TEXT NOT NULL,
    product TEXT NOT NULL,
    units INTEGER NOT NULL,
    unit_price NUMERIC(10,2) NOT NULL,
    revenue NUMERIC(12,2) NOT NULL
);

-- Dimension-like table for targets (useful for Power BI KPI measures)
CREATE TABLE IF NOT EXISTS sales_targets (
    month_start DATE NOT NULL,
    country TEXT NOT NULL,
    target_revenue NUMERIC(12,2) NOT NULL
);

-- Seed a small initial dataset (students can scale it later)
INSERT INTO sales (sale_date, country, channel, product_category, product, units, unit_price, revenue) VALUES
('2024-01-01','France','Retail','Watches','Aurum One',3,4200.00,12600.00),
('2024-01-02','France','Online','Jewelry','Luna Ring',6,650.00,3900.00),
('2024-01-03','Italy','Retail','Jewelry','Luna Ring',9,650.00,5850.00),
('2024-01-04','USA','Online','Watches','Aurum One',2,4200.00,8400.00),
('2024-01-05','Japan','Retail','Watches','Kintsugi Chrono',1,9800.00,9800.00),
('2024-01-06','France','Retail','Accessories','Silk Strap',15,120.00,1800.00),
('2024-01-07','Italy','Online','Watches','Aurum One',1,4200.00,4200.00),
('2024-01-08','Spain','Online','Jewelry','Sol Necklace',4,1200.00,4800.00);

INSERT INTO sales_targets (month_start, country, target_revenue) VALUES
('2024-01-01','France',50000.00),
('2024-01-01','Italy',35000.00),
('2024-01-01','USA',60000.00),
('2024-01-01','Japan',30000.00);
```

</details>

## 📄 `docker-compose.yml`

### What it does  
Defines **how to run the whole project** (multiple containers) with one command.

### Why it’s important  
- Reproducibility: every student gets the same services & versions  
- Isolation: no “it works on my machine”  
- Realistic BI pattern: a DB + an API layer is very common in companies

### Things students usually edit  
- Change ports (`5432`, `18000`) if they conflict locally  
- Add an `.env` file and reference it instead of inline credentials  
- Add new services (e.g., `metabase`, `pgadmin`, `redis`) for extensions


### File content (for reference)

<details><summary>Click to expand</summary>

```yaml
services:
  postgres:
    image: postgres:15
    container_name: bi_postgres
    environment:
      POSTGRES_DB: sales
      POSTGRES_USER: admin
      POSTGRES_PASSWORD: admin
    ports:
      - "5432:5432"
    volumes:
      - pgdata:/var/lib/postgresql/data
      - ./database/init.sql:/docker-entrypoint-initdb.d/init.sql:ro
    healthcheck:
      test: ["CMD-SHELL", "pg_isready -U admin -d sales"]
      interval: 5s
      timeout: 3s
      retries: 20

  api:
    build: ./api
    container_name: bi_api
    environment:
      DB_HOST: postgres
      DB_NAME: sales
      DB_USER: admin
      DB_PASSWORD: admin
      DB_PORT: "5432"
    ports:
      - "18000:8000"
    depends_on:
      postgres:
        condition: service_healthy
    volumes:
      - ./:/workspaces/bi-docker-workshop

volumes:
  pgdata:
```

</details>

## 📄 `docs/bigquery_extension.md`

### What it does  
Extra documentation for workshop-specific topics.

### Why it’s important  
Keeps the main README small, and allows deeper dives without clutter.


### File content (for reference)

<details><summary>Click to expand</summary>

```markdown
# BigQuery Extension (Optional)

Goal: teach a realistic “Modern Data Stack” pattern:
- **Docker** runs ingestion/transforms (compute)
- **BigQuery** stores curated datasets (warehouse)
- **Power BI** consumes BigQuery (semantic/visualization)

## Suggested flow (1 hour extension)
1. Create a BigQuery dataset (e.g., `bi_workshop`)
2. Create a service account with BigQuery write permissions
3. From a container (or local Python), extract from Postgres and load into BigQuery

### Student task idea
- Add a `dim_date` table
- Create an aggregated table `sales_daily_country`
- Load it into BigQuery
- Connect Power BI to BigQuery and build KPI cards + trends

> Keep secrets in `.env` and NEVER commit them.
```

</details>

## 📄 `docs/devcontainer.md`

### What it does  
Extra documentation for workshop-specific topics.

### Why it’s important  
Keeps the main README small, and allows deeper dives without clutter.


### File content (for reference)

<details><summary>Click to expand</summary>

```markdown
# VS Code Dev Containers (optional)

If students have the Dev Containers extension, they can open this repo in a containerized dev environment.

## Steps
1. Install VS Code extension: **Dev Containers**
2. Open the repo in VS Code
3. Command Palette → **Dev Containers: Reopen in Container**

This will attach VS Code to the `api` service container.

## Notes
- The database is still in the `postgres` service
- Use host `postgres` from inside the container
```

</details>

## 📄 `notebook.ipynb`

### What it does  
An existing notebook included in the repo (often used for demos, API tests, or student exercises).

### Why it’s important  
Notebooks are great for:
- quick API calls (`requests`)
- data exploration (`pandas`)
- documenting experiments inside the repo


### File content (for reference)

<details><summary>Click to expand</summary>

```json
{
 "cells": [
  {
   "cell_type": "code",
   "execution_count": 1,
   "id": "55a5a1a3",
   "metadata": {},
   "outputs": [
    {
     "data": {
      "text/plain": [
       "['/usr/local/lib/python311.zip',\n",
       " '/usr/local/lib/python3.11',\n",
       " '/usr/local/lib/python3.11/lib-dynload',\n",
       " '',\n",
       " '/root/.local/lib/python3.11/site-packages',\n",
       " '/usr/local/lib/python3.11/site-packages']"
      ]
     },
     "execution_count": 1,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "import sys\n",
    "sys.path"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 2,
   "id": "8e5fb3f0",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "/usr/local/bin/python\n"
     ]
    }
   ],
   "source": [
    "!which python"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 3,
   "id": "76265b26",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "Python 3.11.14\n"
     ]
    }
   ],
   "source": [
    "!python --version"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 4,
   "id": "54e772da",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "annotated-types==0.7.0\n",
      "anyio==4.12.1\n",
      "asttokens==3.0.1\n",
      "click==8.3.1\n",
      "comm==0.2.3\n",
      "debugpy==1.8.20\n",
      "decorator==5.2.1\n",
      "executing==2.2.1\n",
      "fastapi==0.115.6\n",
      "h11==0.16.0\n",
      "httptools==0.7.1\n",
      "idna==3.11\n",
      "ipykernel==7.2.0\n",
      "ipython==9.10.0\n",
      "ipython_pygments_lexers==1.1.1\n",
      "jedi==0.19.2\n",
      "jupyter_client==8.8.0\n",
      "jupyter_core==5.9.1\n",
      "matplotlib-inline==0.2.1\n",
      "nest-asyncio==1.6.0\n",
      "numpy==2.4.2\n",
      "packaging==26.0\n",
      "pandas==2.2.3\n",
      "parso==0.8.5\n",
      "pexpect==4.9.0\n",
      "platformdirs==4.5.1\n",
      "prompt_toolkit==3.0.52\n",
      "psutil==7.2.2\n",
      "psycopg2-binary==2.9.9\n",
      "ptyprocess==0.7.0\n",
...
(Truncated in notebook — open the file in VS Code to read the full content.)
```

</details>

# 4) Git metadata files (`.git/…`)

These files are **generated by Git** and usually not edited during the workshop. They are included here because the archive contains them, and you asked for *every file*.

In a normal shared repository, `.git/` is **not** committed (students will have their own `.git/` after cloning).

| File | What it is |
|---|---|
| `.git/HEAD` | Points to the current branch ref |
| `.git/config` | Local repo configuration (remotes, settings) |
| `.git/description` | Optional description (rarely used outside GitWeb) |
| `.git/hooks/applypatch-msg.sample` | Template hook script (inactive until copied/renamed) |
| `.git/hooks/commit-msg.sample` | Template hook script (inactive until copied/renamed) |
| `.git/hooks/fsmonitor-watchman.sample` | Template hook script (inactive until copied/renamed) |
| `.git/hooks/post-update.sample` | Template hook script (inactive until copied/renamed) |
| `.git/hooks/pre-applypatch.sample` | Template hook script (inactive until copied/renamed) |
| `.git/hooks/pre-commit.sample` | Template hook script (inactive until copied/renamed) |
| `.git/hooks/pre-merge-commit.sample` | Template hook script (inactive until copied/renamed) |
| `.git/hooks/pre-push.sample` | Template hook script (inactive until copied/renamed) |
| `.git/hooks/pre-rebase.sample` | Template hook script (inactive until copied/renamed) |
| `.git/hooks/pre-receive.sample` | Template hook script (inactive until copied/renamed) |
| `.git/hooks/prepare-commit-msg.sample` | Template hook script (inactive until copied/renamed) |
| `.git/hooks/push-to-checkout.sample` | Template hook script (inactive until copied/renamed) |
| `.git/hooks/update.sample` | Template hook script (inactive until copied/renamed) |
| `.git/info/exclude` | Local ignore rules (like .gitignore but not shared) |

## 📄 `.git/HEAD` (peek)

<details><summary>Click to expand</summary>

```text
ref: refs/heads/master
```

</details>

## 📄 `.git/config` (peek)

<details><summary>Click to expand</summary>

```text
[core]
	repositoryformatversion = 0
	filemode = false
	bare = false
	logallrefupdates = true
	symlinks = false
	ignorecase = true
```

</details>

## 📄 `.git/hooks/pre-commit.sample` (peek)

<details><summary>Click to expand</summary>

```text
#!/bin/sh
#
# An example hook script to verify what is about to be committed.
# Called by "git commit" with no arguments.  The hook should
# exit with non-zero status after issuing an appropriate message if
# it wants to stop the commit.
#
# To enable this hook, rename this file to "pre-commit".

if git rev-parse --verify HEAD >/dev/null 2>&1
then
	against=HEAD
else
	# Initial commit: diff against an empty tree object
	against=$(git hash-object -t tree /dev/null)
fi

# If you want to allow non-ASCII filenames set this variable to true.
allownonascii=$(git config --type=bool hooks.allownonascii)

# Redirect output to stderr.
exec 1>&2

# Cross platform projects tend to avoid non-ASCII filenames; prevent
# them from being added to the repository. We exploit the fact that the
# printable range starts at the space character and ends with tilde.
if [ "$allownonascii" != "true" ] &&
	# Note that the use of brackets around a tr range is ok here, (it's
	# even required, for portability to Solaris 10's /usr/bin/tr), since
	# the square bracket bytes happen to fall in the designated range.
	test $(git diff --cached --name-only --diff-filter=A -z $against |
	  LC_ALL=C tr -d '[ -~]\0' | wc -c
...
(Truncated in notebook — open the file in VS Code to read the full content.)
```

</details>

# 5) Suggested student exercises (hands-on)

1. **Change the dataset**
   - Add a new column in `sales` (e.g., `brand`, `discount_rate`)
   - Update `database/init.sql`
   - Recreate the DB volume and re-run

2. **Add a new API endpoint**
   - Create `/kpi/revenue_by_month`
   - Use `GROUP BY date_trunc('month', sale_date)` in SQL

3. **Power BI integration**
   - Connect to `http://localhost:18000/sales`
   - Create visuals + measures
   - Bonus: call `/kpi/revenue_by_country` as a separate query

4. **Production-hardening (discussion)**
   - Move credentials to `.env` and reference with `env_file:` in Compose
   - Add `uvicorn` command + proper logging
   - Add `pgadmin` service for DB admin
